# Getting Started with GX Core

We will learn how to:

- connect to data
- create expectations
- validate data
- review validation results
- publish and browse the results as **Data Docs**

First of all, let's import the core library of Great Expectations.

Specifically, the `great_expectations` module is the root of the GX Core library.</br>
It contains shortcuts and convenience methods for starting a GX project in a Python session.


In [ ]:
import great_expectations as gx
import pandas as pd

## Importing the data

For this lab, we will be using a sample of [NYC taxi trip record data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

> Yellow and green **taxi trip records** include fields capturing _pick-up and drop-off dates/times_, _pick-up and drop-off locations_, _trip distances_, _itemized fares_, _rate types_, _payment types_, and driver-reported _passenger counts_. The data used in the attached datasets were collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers authorized under the Taxicab & Livery Passenger Enhancement Programs (TPEP/LPEP).

Let's download and read the sample data into a Pandas DataFrame.


In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/great-expectations/gx_tutorials/main/data/yellow_tripdata_sample_2019-01.csv"
)
df.head()

### Create a **Data Context**

A **Data Context** object serves as the entrypoint for interacting with GX components.

We create a **File Data Context**. It stores its configuration, Expectation Suites, Validation Definitions, Checkpoints and Data Docs in a `gx/` folder next to this notebook, so they persist across sessions. (By contrast, an *ephemeral* Data Context, the default when no `gx/` folder exists, keeps everything in memory and loses it when the kernel stops.)

> Because the context persists, every GX object below is created with an `add_or_update` method. That way you can re-run the notebook without "already exists" errors.

In [ ]:
context = gx.get_context(mode="file", project_root_dir=".")
print(context.root_directory)

### Connect to data and create a **Batch**

Let's define:

- a **Data Source**
- a **Data Asset**
- a **Batch Definition**
- a **Batch**


In [ ]:
data_source = context.data_sources.add_or_update_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

### Create an **Expectation**

We expect that the contents of the column `passenger_count` consist of values ranging from 1 to 6.


In [ ]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="passenger_count", min_value=1, max_value=6
)

### Validate the data batch

Using the expectation you just created, validate the data batch.


In [ ]:
validation_result = batch.validate(expectation)

In [ ]:
print(validation_result)

### Create an **Expectation Suite**


In [ ]:
suite = context.suites.add_or_update(
    gx.ExpectationSuite(
        name="Taxi trip expectations",
    )
)

In [ ]:
# Expectation 1
suite.add_expectation(expectation)

# Expectation 2
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="fare_amount", min_value=0)
)

# Expectation 3
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="pickup_datetime")
)

# Expectation 4
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="payment_type", type_="int")
)

# Expectation 5
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="payment_type")
)

# Expectation 6
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="payment_type", value_set=[1, 2, 3, 4]
    )
)


### Create a **Validation Definition**

A **Validation Definition** explicitly ties together the Batch of data to be validated to the Expectation Suite that should be used to validate it.


In [ ]:
validation_definition = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(
        name="Validation definition", data=batch_definition, suite=suite
    )
)

### Configure a **Data Docs** site

**Data Docs** are human-readable HTML pages that GX renders from your Expectation Suites and Validation Results. They are a shared report of what you expect from the data and whether it meets those expectations. The whole team can browse them, and you can publish them as a static website.

A Data Context can have one or more *Data Docs sites*. A File Data Context already comes with a default site called `local_site`. To see how a site is configured, we add our own, `lab_site`, whose HTML files are written under `gx/uncommitted/data_docs/lab_site/`.

In [ ]:
site_name = "lab_site"
site_config = {
    "class_name": "SiteBuilder",
    "site_index_builder": {"class_name": "DefaultSiteIndexBuilder"},
    "store_backend": {
        "class_name": "TupleFilesystemStoreBackend",
        # path relative to the root folder of the Data Context (gx/)
        "base_directory": "uncommitted/data_docs/lab_site/",
    },
}

# Site names must be unique: add the site only on the first run
if site_name not in context.list_data_docs_sites():
    context.add_data_docs_site(site_name=site_name, site_config=site_config)

context.list_data_docs_sites()

### Create and run a **Checkpoint**

Create and run a Checkpoint to validate the data based on the supplied Validation Definition.

A Checkpoint can also run **Actions** after validation, such as sending a Slack or email notification. Here we attach an `UpdateDataDocsAction` so the Data Docs site is rebuilt with the new Validation Results every time the Checkpoint runs.

In [ ]:
checkpoint = context.checkpoints.add_or_update(
    gx.Checkpoint(
        name="checkpoint",
        validation_definitions=[validation_definition],
        actions=[
            gx.checkpoint.actions.UpdateDataDocsAction(
                name="update_data_docs", site_names=[site_name]
            )
        ],
    )
)
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})
print(checkpoint_result.describe())

### Explore the **Data Docs**

Open the Data Docs site in your browser. If no browser tab opens, copy the URL printed below into your browser.

In [ ]:
context.open_data_docs(site_name=site_name)
context.get_docs_sites_urls(site_names=[site_name])

In the Data Docs:

1. On the home page, find the **Validation Results** list and the **Expectation Suites** list.
2. Open the latest validation run of `Taxi trip expectations`. Which Expectation failed?
3. Look at the failed Expectation: what percentage of rows was unexpected, and which values does GX show as a sample?
4. Open the `Taxi trip expectations` suite page. How does GX describe each Expectation in plain English?

**Exercise:** add a new Expectation to the suite (pick one from the [Expectations Gallery](https://greatexpectations.io/expectations/)). Then re-run the Validation Definition and Checkpoint cells, and refresh the Data Docs. The new run appears next to the previous ones.

> Data Docs can also be rebuilt manually, without running a Checkpoint, with `context.build_data_docs(site_names=site_name)`.

---

## References

[1] https://docs.greatexpectations.io/docs/core/introduction/try_gx/

[2] [Expectations Gallery](https://greatexpectations.io/expectations/)

[3] [Configure Data Docs](https://docs.greatexpectations.io/docs/core/configure_project_settings/configure_data_docs/)

[4] [Create a Checkpoint with Actions](https://docs.greatexpectations.io/docs/core/trigger_actions_based_on_results/create_a_checkpoint_with_actions/)